# Experiment 8: Deploy Docker Container to Cloud VM (Amazon EC2)

**Objective:**
- Set up an AWS EC2 instance
- Deploy the Docker container to EC2
- Expose the service using a public IP

**Prerequisites:** AWS account, Docker image built (Expt 5), SSH key pair

## Step 1: Install AWS CLI

In [ ]:
# Install AWS CLI and boto3
!pip install awscli boto3

import subprocess
result = subprocess.run(['aws', '--version'], capture_output=True, text=True)
print(f"AWS CLI: {result.stdout.strip()}")

## Step 2: AWS Configuration

Run this in your terminal:
```bash
aws configure
```

Enter:
- AWS Access Key ID
- AWS Secret Access Key
- Default region: us-east-1
- Default output: json

In [ ]:
import boto3
import json
import time
import os

# Configuration
AWS_REGION = "us-east-1"
INSTANCE_TYPE = "t2.micro"  # Free tier eligible
KEY_NAME = "mlops-key"  # SSH key pair name
SECURITY_GROUP_NAME = "mlops-sg"
DOCKER_IMAGE = "churn-prediction-api:latest"

# Initialize AWS clients
ec2_client = boto3.client('ec2', region_name=AWS_REGION)
ec2_resource = boto3.resource('ec2', region_name=AWS_REGION)

print(f"AWS Region: {AWS_REGION}")
print(f"Instance Type: {INSTANCE_TYPE}")
print("AWS clients initialized!")

## Step 3: Create Key Pair

In [ ]:
# Create SSH key pair
try:
    key_pair = ec2_client.create_key_pair(KeyName=KEY_NAME, KeyType='rsa')
    
    # Save private key
    os.makedirs('aws_keys', exist_ok=True)
    key_path = f'aws_keys/{KEY_NAME}.pem'
    with open(key_path, 'w') as f:
        f.write(key_pair['KeyMaterial'])
    os.chmod(key_path, 0o400)
    
    print(f"Key pair '{KEY_NAME}' created!")
    print(f"Private key saved to: {key_path}")
except Exception as e:
    if 'InvalidKeyPair.Duplicate' in str(e):
        print(f"Key pair '{KEY_NAME}' already exists")
    else:
        print(f"Error: {e}")

## Step 4: Create Security Group

In [ ]:
try:
    # Get default VPC
    vpcs = ec2_client.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
    vpc_id = vpcs['Vpcs'][0]['VpcId']
    
    # Create security group
    sg = ec2_client.create_security_group(
        GroupName=SECURITY_GROUP_NAME,
        Description='Security group for MLOps churn prediction API',
        VpcId=vpc_id
    )
    sg_id = sg['GroupId']
    
    # Add inbound rules
    ec2_client.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            {'IpProtocol': 'tcp', 'FromPort': 22, 'ToPort': 22, 'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'SSH'}]},
            {'IpProtocol': 'tcp', 'FromPort': 8000, 'ToPort': 8000, 'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'API'}]},
            {'IpProtocol': 'tcp', 'FromPort': 80, 'ToPort': 80, 'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'HTTP'}]},
            {'IpProtocol': 'tcp', 'FromPort': 443, 'ToPort': 443, 'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'HTTPS'}]}
        ]
    )
    print(f"Security group '{SECURITY_GROUP_NAME}' created: {sg_id}")
    print("Inbound rules: SSH(22), API(8000), HTTP(80), HTTPS(443)")
    
except Exception as e:
    if 'InvalidGroup.Duplicate' in str(e):
        sgs = ec2_client.describe_security_groups(GroupNames=[SECURITY_GROUP_NAME])
        sg_id = sgs['SecurityGroups'][0]['GroupId']
        print(f"Security group already exists: {sg_id}")
    else:
        print(f"Error: {e}")
        sg_id = None

## Step 5: Launch EC2 Instance

In [ ]:
# User data script to install Docker and run the container
user_data_script = """#!/bin/bash
set -e

# Update system
sudo yum update -y

# Install Docker
sudo yum install -y docker
sudo systemctl start docker
sudo systemctl enable docker
sudo usermod -aG docker ec2-user

# Pull and run the Docker container
# Replace with your Docker Hub image
# sudo docker pull YOUR_DOCKERHUB_USERNAME/churn-prediction-api:latest
# sudo docker run -d --name churn-api -p 8000:8000 \\
#   -e SECRET_KEY=mlops-secret-key-2024 \\
#   -e API_KEYS=mlops-api-key-001,mlops-api-key-002 \\
#   --restart unless-stopped \\
#   YOUR_DOCKERHUB_USERNAME/churn-prediction-api:latest

echo "Setup completed!" > /home/ec2-user/setup_complete.txt
"""

print("User data script prepared")
print("This script will install Docker on the EC2 instance")

In [ ]:
# Get the latest Amazon Linux 2023 AMI
amis = ec2_client.describe_images(
    Filters=[
        {'Name': 'name', 'Values': ['al2023-ami-2023.*-x86_64']},
        {'Name': 'state', 'Values': ['available']}
    ],
    Owners=['amazon']
)

# Sort by creation date and get the latest
sorted_amis = sorted(amis['Images'], key=lambda x: x['CreationDate'], reverse=True)
ami_id = sorted_amis[0]['ImageId']
print(f"Latest AMI: {ami_id}")
print(f"AMI Name: {sorted_amis[0]['Name']}")

In [ ]:
# Launch EC2 instance
try:
    instances = ec2_resource.create_instances(
        ImageId=ami_id,
        InstanceType=INSTANCE_TYPE,
        KeyName=KEY_NAME,
        SecurityGroupIds=[sg_id],
        MinCount=1,
        MaxCount=1,
        UserData=user_data_script,
        TagSpecifications=[
            {
                'ResourceType': 'instance',
                'Tags': [
                    {'Key': 'Name', 'Value': 'mlops-churn-api'},
                    {'Key': 'Project', 'Value': 'MLOps'},
                    {'Key': 'Environment', 'Value': 'production'}
                ]
            }
        ]
    )
    
    instance = instances[0]
    instance_id = instance.id
    print(f"EC2 Instance launched: {instance_id}")
    print("Waiting for instance to be running...")
    
    instance.wait_until_running()
    instance.reload()
    
    public_ip = instance.public_ip_address
    print(f"\nInstance is running!")
    print(f"Instance ID: {instance_id}")
    print(f"Public IP: {public_ip}")
    print(f"\nSSH Command: ssh -i aws_keys/{KEY_NAME}.pem ec2-user@{public_ip}")
    print(f"API Endpoint: http://{public_ip}:8000")
    
except Exception as e:
    print(f"Error launching instance: {e}")

## Step 6: Deploy Application to EC2

In [ ]:
# Generate deployment script
deploy_script = f"""#!/bin/bash
# ============================================================
# DEPLOYMENT SCRIPT FOR EC2
# ============================================================
# Run this script on the EC2 instance

set -e

# Variables
DOCKER_IMAGE="YOUR_DOCKERHUB_USERNAME/churn-prediction-api:latest"
CONTAINER_NAME="churn-api"

echo "=== Starting Deployment ==="

# Stop existing container
echo "Stopping existing container..."
docker stop $CONTAINER_NAME 2>/dev/null || true
docker rm $CONTAINER_NAME 2>/dev/null || true

# Pull latest image
echo "Pulling latest image..."
docker pull $DOCKER_IMAGE

# Run new container
echo "Starting new container..."
docker run -d \\
  --name $CONTAINER_NAME \\
  -p 8000:8000 \\
  -e SECRET_KEY=mlops-secret-key-2024 \\
  -e API_KEYS=mlops-api-key-001,mlops-api-key-002 \\
  --restart unless-stopped \\
  $DOCKER_IMAGE

# Wait and verify
sleep 5
echo "Container status:"
docker ps --filter name=$CONTAINER_NAME

echo "Health check:"
curl -s http://localhost:8000/health || echo "Health check failed"

echo ""
echo "=== Deployment Complete ==="
"""

with open('deploy.sh', 'w') as f:
    f.write(deploy_script)

os.chmod('deploy.sh', 0o755)
print("Deployment script created: deploy.sh")
print("\nManual deployment steps:")
print(f"  1. scp -i aws_keys/{KEY_NAME}.pem deploy.sh ec2-user@<EC2_IP>:~/")
print(f"  2. ssh -i aws_keys/{KEY_NAME}.pem ec2-user@<EC2_IP>")
print(f"  3. chmod +x deploy.sh && ./deploy.sh")

## Step 7: Test Deployed API

In [ ]:
import requests

# Replace with your EC2 public IP
EC2_PUBLIC_IP = "YOUR_EC2_PUBLIC_IP"  # e.g., "54.123.45.67"
EC2_API_URL = f"http://{EC2_PUBLIC_IP}:8000"

try:
    # Health check
    print(f"Testing API at {EC2_API_URL}")
    response = requests.get(f"{EC2_API_URL}/health", timeout=10)
    print(f"Health: {response.json()}")
    
    # Prediction test
    response = requests.post(
        f"{EC2_API_URL}/predict",
        json={
            "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
            "MonthlyCharges": 70.5, "Contract": "One year",
            "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
        },
        headers={"X-API-Key": "mlops-api-key-001"},
        timeout=10
    )
    print(f"Prediction: {json.dumps(response.json(), indent=2)}")
    print("\n✅ EC2 deployment verified!")
    
except requests.exceptions.ConnectionError:
    print("Cannot connect to EC2. Ensure:")
    print("  1. EC2 instance is running")
    print("  2. Security group allows port 8000")
    print("  3. Docker container is running on EC2")
    print(f"  4. Replace EC2_PUBLIC_IP with your actual IP")
except Exception as e:
    print(f"Error: {e}")

## Step 8: Cleanup (Optional)

In [ ]:
# CAUTION: Uncomment to terminate EC2 instance
# This will permanently destroy the instance!

# instance_id = "i-xxxxxxxxxxxxxxxxx"  # Replace with your instance ID
# ec2_client.terminate_instances(InstanceIds=[instance_id])
# print(f"Instance {instance_id} termination initiated")

print("\nCleanup commands (run manually):")
print("  aws ec2 terminate-instances --instance-ids <INSTANCE_ID>")
print("  aws ec2 delete-security-group --group-id <SG_ID>")
print("  aws ec2 delete-key-pair --key-name mlops-key")